# IFC → Turtle (LBD) Konvertierung
Konvertiert `data/gantenbein/FoC_Preserving_IFC-Gantenbein.ifc` über das Node.js CLI-Tool `ifc-lbd` nach Turtle.

**Ablauf:**
1. Node.js CLI erzeugt eine JSON-LD-Datei (alle Subsets: BOT, FSO, Products, Properties)
2. `rdflib` lädt das JSON-LD und serialisiert es als Turtle (`.ttl`)

In [ ]:
INST_NS      = "https://dca.ethz.ch/gantenbein/"  # IFC-Objekte
SCHEMA_NS    = "https://dca.ethz.ch/prop/"          # Schema-Properties
SCHEMA_PREFIX = "dca-prop"                          # Präfix-Name in der TTL

In [ ]:
import subprocess
import sys
from pathlib import Path

from rdflib import Graph, Namespace

# --- Pfade ---
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IFC_FILE     = PROJECT_ROOT / "data" / "gantenbein" / "FoC_Preserving_IFC-Gantenbein.ifc"
CLI          = PROJECT_ROOT / "dist" / "cli-index.cjs"
OUTPUT_DIR   = PROJECT_ROOT / "output"
JSONLD_OUT   = OUTPUT_DIR / "gantenbein.jsonld"
TTL_OUT      = OUTPUT_DIR / "gantenbein.ttl"

OUTPUT_DIR.mkdir(exist_ok=True)

print(f"IFC:    {IFC_FILE}  (exists: {IFC_FILE.exists()})")
print(f"CLI:    {CLI}  (exists: {CLI.exists()})")
print(f"Output: {OUTPUT_DIR}")

IFC:    /Users/padrian/Documents/08_Tools/IFC-LBD/data/gantenbein/FoC_Preserving_IFC-Gantenbein.ifc  (exists: True)
CLI:    /Users/padrian/Documents/08_Tools/IFC-LBD/dist/cli-index.cjs  (exists: True)
Output: /Users/padrian/Documents/08_Tools/IFC-LBD/output


## Schritt 1 — IFC → JSON-LD via Node.js CLI

Subset `all` erzeugt BOT + FSO + Products + Properties.  
Für nur BOT/FSO: `"bot"` oder `"fso"` als ersten Parameter übergeben.

In [ ]:
SUBSET    = "all"  # Optionen: bot | fso | products | properties | all

cmd = [
    "node",
    str(CLI),
    SUBSET,
    "-i", str(IFC_FILE),
    "-o", str(JSONLD_OUT),
    "-f", "jsonld",
    "-n", INST_NS,
    "-e", SCHEMA_NS,
]

print("Starte Konvertierung …")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("FEHLER:", result.stderr)
    sys.exit(1)

print(result.stdout or "(kein stdout)")
print(f"JSON-LD gespeichert: {JSONLD_OUT}  ({JSONLD_OUT.stat().st_size / 1_000_000:.1f} MB)")

Starte Konvertierung …
INFO:  Parsing Model using IFC4 Schema
Finished FSO parsing: 4.767ms

Finished products parsing: 30.375ms
Finished BOT parsing: 37.424ms


Finished PROPERTIES parsing: 43.446ms

JSON-LD gespeichert: /Users/padrian/Documents/08_Tools/IFC-LBD/output/gantenbein.jsonld  (0.1 MB)


## Schritt 2 — JSON-LD → Turtle via rdflib

In [ ]:
print("Lade JSON-LD in rdflib …")
g = Graph()
g.parse(str(JSONLD_OUT), format="json-ld")
g.bind(SCHEMA_PREFIX, Namespace(SCHEMA_NS), override=True)

print(f"Triples geladen: {len(g):,}")

Lade JSON-LD in rdflib …
Triples geladen: 882
Turtle gespeichert: /Users/padrian/Documents/08_Tools/IFC-LBD/output/gantenbein.ttl  (0.0 MB)


## Schritt 2b — IFC-Hierarchie ergänzen (via ifcopenshell)

`web-ifc` kann die Beziehungen für dieses Modell nicht lesen, da die GlobalId-Felder in den 
`IFCRELAGGREGATES`- und `IFCRELCONTAINEDINSPATIALSTRUCTURE`-Einträgen leer sind (`$`). 
`ifcopenshell` liest diese korrekt und ergänzt:
- `bot:hasBuilding` / `bot:hasStorey` / `bot:hasSpace` (Zonenhierarchie)
- `bot:containsElement` (Stockwerk → Elemente)

In [ ]:
import ifcopenshell
from rdflib import URIRef
from urllib.parse import quote

BOT = Namespace("https://w3id.org/bot#")

def inst_uri(global_id: str) -> URIRef:
    return URIRef(INST_NS + quote(global_id, safe=''))

ifc = ifcopenshell.open(str(IFC_FILE))
added = 0

# Zonenhierarchie: Site → Building → Storey → Space
for site in ifc.by_type("IfcSite"):
    site_uri = inst_uri(site.GlobalId)
    for rel in site.IsDecomposedBy:
        for bldg in rel.RelatedObjects:
            bldg_uri = inst_uri(bldg.GlobalId)
            g.add((site_uri, BOT.hasBuilding, bldg_uri))
            added += 1
            for rel2 in bldg.IsDecomposedBy:
                for storey in rel2.RelatedObjects:
                    storey_uri = inst_uri(storey.GlobalId)
                    g.add((bldg_uri, BOT.hasStorey, storey_uri))
                    added += 1
                    for rel3 in storey.IsDecomposedBy:
                        for space in rel3.RelatedObjects:
                            space_uri = inst_uri(space.GlobalId)
                            g.add((storey_uri, BOT.hasSpace, space_uri))
                            added += 1

# Stockwerk → Elemente: bot:containsElement
for storey in ifc.by_type("IfcBuildingStorey"):
    storey_uri = inst_uri(storey.GlobalId)
    for rel in storey.ContainsElements:
        for elem in rel.RelatedElements:
            g.add((storey_uri, BOT.containsElement, inst_uri(elem.GlobalId)))
            added += 1

# Space → Elemente: bot:containsElement
for space in ifc.by_type("IfcSpace"):
    space_uri = inst_uri(space.GlobalId)
    for rel in space.ContainsElements:
        for elem in rel.RelatedElements:
            g.add((space_uri, BOT.containsElement, inst_uri(elem.GlobalId)))
            added += 1

print(f"Ergänzte Triples: {added}")
print(f"Gesamte Triples: {len(g):,}")

## Schritt 2c — TTL speichern

In [ ]:
g.serialize(destination=str(TTL_OUT), format="turtle")
print(f"Turtle gespeichert: {TTL_OUT}  ({TTL_OUT.stat().st_size / 1_000_000:.1f} MB)")

## Schritt 3 — Kurze Validierung

In [ ]:
from rdflib.namespace import RDF
from rdflib import URIRef, Namespace

BOT = Namespace("https://w3id.org/bot#")
FSO = Namespace("https://w3id.org/fso#")

storeys  = list(g.subjects(RDF.type, BOT.Storey))
spaces   = list(g.subjects(RDF.type, BOT.Space))
elements = list(g.subjects(RDF.type, BOT.Element))

print(f"BOT:Storey   {len(storeys):>6}")
print(f"BOT:Space    {len(spaces):>6}")
print(f"BOT:Element  {len(elements):>6}")

# Erste 5 Stockwerke
print("\nStoreys:")
for s in storeys[:5]:
    label = g.value(s, URIRef("http://www.w3.org/2000/01/rdf-schema#label"))
    print(f"  {s.split('#')[-1]}  {label or ''}")

BOT:Storey        3
BOT:Space         0
BOT:Element     173

Storeys:
  https://web-bim/resources/0922xsxjr7Kf7YumLk%24s4S  Basement
  https://web-bim/resources/1KLoJ2lZH9FRD2JiAQMRIt  First Floor
  https://web-bim/resources/1m8JeOBhH40w5xPgClN80d  Ground Floor
